# Activity No. 1: Machine Learning Classification and Regression

| | |
|---|---|
| **Name** | Yash Galande |
| **PRN** | 202401040179 |
| **Batch** | A2 |
| **Roll No.** | 41 |
| **Application** | Residential Property (Flat) Price Analysis |
| **Dataset** | Synthetic flat-listing dataset generated in this notebook |
| **GitHub** | [View repository](https://github.com/PASTE-YOUR-LINK-HERE) |

## Aim
To implement and compare classification and regression algorithms for analysing residential flat prices.

## Problem Statement
The price of a flat depends on its carpet area, number of bedrooms, age of the building, floor number, distance from the city centre, parking spaces, furnishing level and locality tier. The task is to:
1. **Classify** a flat into a price segment (*Affordable*, *Mid-range*, *Luxury*), and
2. **Predict** its continuous price (in lakh rupees).

## Objectives
1. Generate and understand a structured housing dataset.
2. Explore distributions, correlations and missing values.
3. Preprocess numerical and categorical features using a pipeline.
4. Split data into training and testing sets (stratified).
5. Train Logistic Regression, KNN, Decision Tree and Random Forest classifiers.
6. Evaluate using accuracy, precision, recall, F1-score and confusion matrices.
7. Train Linear Regression, Ridge, Decision Tree and Random Forest regressors.
8. Evaluate using MAE, MSE, RMSE and R².
9. Cross-validate, compare models, inspect feature importance, and predict for a new flat.

## Theory
* **Classification** predicts a discrete class (price segment); **Regression** predicts a continuous value (price in lakh ₹).
* **Logistic Regression** – linear model giving class probabilities. **KNN** – classifies by majority vote of nearest neighbours. **Decision Tree** – rule-based splits. **Random Forest** – ensemble of many trees (reduces variance).
* **Linear / Ridge Regression** – linear relationship (Ridge adds L2 penalty). Tree models capture non-linear relationships.
* **Preprocessing** – numeric features are standardised; categorical features are one-hot encoded inside a `Pipeline`, so test data never leaks into training.
* **Metrics** – Accuracy, Precision, Recall, F1 (classification); MAE, MSE, RMSE, R² (regression).

> **Dataset note:** the data is synthetic (generated with a fixed seed) to demonstrate the ML workflow; it is not real market data.

## GitHub Repository
🔗 **Source code:** [https://github.com/PASTE-YOUR-LINK-HERE](https://github.com/PASTE-YOUR-LINK-HERE)

## 1. Import required libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold, KFold
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression, LinearRegression, Ridge
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             confusion_matrix, classification_report, ConfusionMatrixDisplay,
                             mean_absolute_error, mean_squared_error, r2_score)

RANDOM_STATE = 42
rng = np.random.default_rng(RANDOM_STATE)
print("Libraries imported successfully.")

## 2. Generate the housing dataset

In [ ]:
n = 2000

area_sqft   = rng.integers(400, 2501, n)
bedrooms    = np.clip(np.round(area_sqft / 650 + rng.normal(0, 0.5, n)), 1, 4).astype(int)
age_years   = rng.integers(0, 31, n)
floor       = rng.integers(0, 26, n)
dist_centre = rng.uniform(1, 30, n).round(1)
parking     = rng.choice([0, 1, 2], n, p=[0.25, 0.55, 0.20])
furnishing  = rng.choice(["Unfurnished", "Semi-furnished", "Furnished"], n, p=[0.40, 0.38, 0.22])
locality    = rng.choice(["Standard", "Prime", "Premium"], n, p=[0.50, 0.35, 0.15])

locality_bonus  = pd.Series(locality).map({"Standard": 0, "Prime": 18, "Premium": 40}).to_numpy()
furnishing_bonus = pd.Series(furnishing).map({"Unfurnished": 0, "Semi-furnished": 3, "Furnished": 6}).to_numpy()

# Synthetic price formula (lakh rupees) + random noise
price = (8 + area_sqft * 0.045 + bedrooms * 4 + locality_bonus + furnishing_bonus
         + parking * 3 + floor * 0.15 - age_years * 0.9 - dist_centre * 1.2
         + rng.normal(0, 8, n))
price = np.clip(price, 10, None).round(2)

df = pd.DataFrame({
    "area_sqft": area_sqft, "bedrooms": bedrooms, "age_years": age_years, "floor": floor,
    "distance_centre_km": dist_centre, "parking_spaces": parking,
    "furnishing": furnishing, "locality_tier": locality, "price_lakh": price
})

# Price segments from price quantiles (40% / 40% / 20%) -> reasonably balanced classes
df["price_segment"] = pd.qcut(df["price_lakh"], q=[0, 0.4, 0.8, 1.0],
                              labels=["Affordable", "Mid-range", "Luxury"])
print("Dataset shape:", df.shape)
display(df.head())

## 3. Explore and validate the dataset

In [ ]:
print("Dataset information:")
df.info()
print("\nMissing values per column:")
display(df.isnull().sum().to_frame("missing_values"))
print("\nDescriptive statistics:")
display(df.describe(include="all").T)
print("\nPrice segment distribution:")
display(df["price_segment"].value_counts())

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 4))

counts = df["price_segment"].value_counts().reindex(["Affordable", "Mid-range", "Luxury"])
axes[0].bar(counts.index.astype(str), counts.values, color=["#4c72b0", "#55a868", "#c44e52"])
axes[0].set(title="Price Segment Distribution", xlabel="Segment", ylabel="Number of flats")

axes[1].hist(df["price_lakh"], bins=30, edgecolor="black", alpha=0.75)
axes[1].set(title="Price Distribution", xlabel="Price (lakh ₹)", ylabel="Frequency")

axes[2].scatter(df["area_sqft"], df["price_lakh"], alpha=0.35, s=12)
axes[2].set(title="Area vs Price", xlabel="Area (sq ft)", ylabel="Price (lakh ₹)")
plt.tight_layout(); plt.show()

# Correlation of numeric features
num_cols = ["area_sqft", "bedrooms", "age_years", "floor", "distance_centre_km", "parking_spaces", "price_lakh"]
corr = df[num_cols].corr()
plt.figure(figsize=(7, 5.5))
plt.imshow(corr, cmap="coolwarm", vmin=-1, vmax=1)
plt.xticks(range(len(num_cols)), num_cols, rotation=45, ha="right")
plt.yticks(range(len(num_cols)), num_cols)
for i in range(len(num_cols)):
    for j in range(len(num_cols)):
        plt.text(j, i, f"{corr.iloc[i, j]:.2f}", ha="center", va="center", fontsize=8)
plt.colorbar(); plt.title("Correlation Heatmap"); plt.tight_layout(); plt.show()

df.boxplot(column="price_lakh", by="locality_tier", figsize=(6, 4))
plt.suptitle(""); plt.title("Price by Locality Tier"); plt.ylabel("Price (lakh ₹)"); plt.show()

## 4. Prepare features and split the data
* Classification target: `price_segment`; regression target: `price_lakh`.
* Both targets are **excluded** from the features to avoid target leakage.
* Stratified 80/20 split keeps class proportions equal in train and test sets.

In [ ]:
numeric_features = ["area_sqft", "bedrooms", "age_years", "floor", "distance_centre_km", "parking_spaces"]
categorical_features = ["furnishing", "locality_tier"]

X = df[numeric_features + categorical_features]
y_class = df["price_segment"].astype(str)
y_reg = df["price_lakh"]

X_train, X_test, yc_train, yc_test, yr_train, yr_test = train_test_split(
    X, y_class, y_reg, test_size=0.20, random_state=RANDOM_STATE, stratify=y_class)

preprocessor = ColumnTransformer([
    ("num", StandardScaler(), numeric_features),
    ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features)
])

print("Training rows:", len(X_train), "| Testing rows:", len(X_test))
print("\nTraining class distribution:")
display(yc_train.value_counts())

## 5. Classification: train and compare models

In [ ]:
classifiers = {
    "Logistic Regression": LogisticRegression(max_iter=2000, random_state=RANDOM_STATE),
    "KNN (k=7)": KNeighborsClassifier(n_neighbors=7),
    "Decision Tree": DecisionTreeClassifier(max_depth=6, random_state=RANDOM_STATE),
    "Random Forest": RandomForestClassifier(n_estimators=200, max_depth=10, random_state=RANDOM_STATE),
}

labels = ["Affordable", "Mid-range", "Luxury"]
clf_rows, trained_clf, clf_preds = [], {}, {}
for name, model in classifiers.items():
    pipe = make_pipeline(preprocessor, model).fit(X_train, yc_train)
    pred = pipe.predict(X_test)
    trained_clf[name], clf_preds[name] = pipe, pred
    clf_rows.append({
        "Model": name,
        "Accuracy": accuracy_score(yc_test, pred),
        "Precision": precision_score(yc_test, pred, average="weighted", zero_division=0),
        "Recall": recall_score(yc_test, pred, average="weighted", zero_division=0),
        "F1 Score": f1_score(yc_test, pred, average="weighted", zero_division=0),
    })

clf_df = pd.DataFrame(clf_rows).sort_values("F1 Score", ascending=False).reset_index(drop=True)
display(clf_df.round(3))

clf_df.set_index("Model")[["Accuracy", "Precision", "Recall", "F1 Score"]].plot(
    kind="bar", figsize=(10, 5), ylim=(0, 1))
plt.title("Classification Model Comparison"); plt.ylabel("Score")
plt.xticks(rotation=15, ha="right"); plt.tight_layout(); plt.show()

In [ ]:
best_clf_name = clf_df.loc[0, "Model"]
best_clf = trained_clf[best_clf_name]
print("Best classifier by F1 score:", best_clf_name)
print("\nClassification report:")
print(classification_report(yc_test, clf_preds[best_clf_name], labels=labels, zero_division=0))

# Confusion matrices for all models
fig, axes = plt.subplots(1, 4, figsize=(20, 4.5))
for ax, (name, pred) in zip(axes, clf_preds.items()):
    ConfusionMatrixDisplay(confusion_matrix(yc_test, pred, labels=labels),
                           display_labels=labels).plot(ax=ax, cmap="Blues", colorbar=False)
    ax.set_title(name); ax.tick_params(axis="x", rotation=20)
plt.tight_layout(); plt.show()

# 5-fold stratified cross-validation (accuracy)
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
cv_clf = {name: cross_val_score(make_pipeline(preprocessor, m), X, y_class, cv=skf, scoring="accuracy")
          for name, m in classifiers.items()}
display(pd.DataFrame({k: {"CV mean accuracy": v.mean(), "CV std": v.std()} for k, v in cv_clf.items()}).T.round(3))

**Classification metrics**
* **Accuracy** – share of all flats placed in the correct segment.
* **Precision** – of the flats predicted in a segment, how many truly belong (weighted average).
* **Recall** – of the flats truly in a segment, how many were found (weighted average).
* **F1-score** – harmonic mean of precision and recall.
* **Confusion matrix** – correct and incorrect predictions per segment.

## 6. Regression: train and compare models

In [ ]:
regressors = {
    "Linear Regression": LinearRegression(),
    "Ridge Regression": Ridge(alpha=1.0),
    "Decision Tree": DecisionTreeRegressor(max_depth=6, random_state=RANDOM_STATE),
    "Random Forest": RandomForestRegressor(n_estimators=200, max_depth=10, random_state=RANDOM_STATE),
}

reg_rows, trained_reg, reg_preds = [], {}, {}
for name, model in regressors.items():
    pipe = make_pipeline(preprocessor, model).fit(X_train, yr_train)
    pred = pipe.predict(X_test)
    trained_reg[name], reg_preds[name] = pipe, pred
    mse = mean_squared_error(yr_test, pred)
    reg_rows.append({"Model": name, "MAE": mean_absolute_error(yr_test, pred),
                     "MSE": mse, "RMSE": np.sqrt(mse), "R2 Score": r2_score(yr_test, pred)})

reg_df = pd.DataFrame(reg_rows).sort_values("RMSE").reset_index(drop=True)
display(reg_df.round(3))

reg_df.set_index("Model")[["MAE", "RMSE"]].plot(kind="bar", figsize=(9, 5))
plt.title("Regression Error Comparison (lower is better)"); plt.ylabel("Error (lakh ₹)")
plt.xticks(rotation=15, ha="right"); plt.tight_layout(); plt.show()

In [ ]:
best_reg_name = reg_df.loc[0, "Model"]
best_reg = trained_reg[best_reg_name]
pred = reg_preds[best_reg_name]
print("Best regressor by lowest RMSE:", best_reg_name)
print("R² score:", round(r2_score(yr_test, pred), 3))

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].scatter(yr_test, pred, alpha=0.5)
lo, hi = min(yr_test.min(), pred.min()), max(yr_test.max(), pred.max())
axes[0].plot([lo, hi], [lo, hi], "r--")
axes[0].set(xlabel="Actual price (lakh ₹)", ylabel="Predicted price (lakh ₹)",
            title=f"Actual vs Predicted — {best_reg_name}")
axes[1].scatter(pred, yr_test - pred, alpha=0.5)
axes[1].axhline(0, color="r", linestyle="--")
axes[1].set(xlabel="Predicted price (lakh ₹)", ylabel="Residual", title="Residual Plot")
plt.tight_layout(); plt.show()

# 5-fold cross-validation (R²)
kf = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
cv_reg = {name: cross_val_score(make_pipeline(preprocessor, m), X, y_reg, cv=kf, scoring="r2")
          for name, m in regressors.items()}
display(pd.DataFrame({k: {"CV mean R2": v.mean(), "CV std": v.std()} for k, v in cv_reg.items()}).T.round(3))

**Regression metrics**
* **MAE** – average absolute difference between actual and predicted price.
* **MSE** – average squared error (penalises large errors).
* **RMSE** – square root of MSE, in lakh rupees.
* **R²** – proportion of price variance explained by the model (can be negative on test data).

## 7. Feature importance (Random Forest)

In [ ]:
rf = trained_reg["Random Forest"]
names = rf.named_steps["columntransformer"].get_feature_names_out()
imp = pd.Series(rf.named_steps["randomforestregressor"].feature_importances_, index=names).sort_values()
imp.plot(kind="barh", figsize=(8, 5), title="Random Forest Feature Importance")
plt.xlabel("Importance"); plt.tight_layout(); plt.show()

## 8. Predict for a new flat

In [ ]:
new_flat = pd.DataFrame([{
    "area_sqft": 1100, "bedrooms": 2, "age_years": 5, "floor": 8,
    "distance_centre_km": 7.5, "parking_spaces": 1,
    "furnishing": "Semi-furnished", "locality_tier": "Prime"
}])

pred_segment = best_clf.predict(new_flat)[0]
pred_price = float(best_reg.predict(new_flat)[0])

print("New flat details:")
display(new_flat)
print("Predicted price segment :", pred_segment)
print(f"Predicted price         : ₹{pred_price:.2f} lakh")

## 9. Conclusion

In [ ]:
from IPython.display import Markdown
c, r = clf_df.iloc[0], reg_df.iloc[0]
display(Markdown(f'''
* **Classification:** Among Logistic Regression, KNN, Decision Tree and Random Forest, **{c["Model"]}** performed best
  (accuracy = {c["Accuracy"]:.3f}, weighted F1 = {c["F1 Score"]:.3f}).
* **Regression:** Among Linear, Ridge, Decision Tree and Random Forest regressors, **{r["Model"]}** gave the lowest error
  (MAE = {r["MAE"]:.2f}, RMSE = {r["RMSE"]:.2f} lakh ₹, R² = {r["R2 Score"]:.3f}).
* For the sample flat, the predicted segment is **{pred_segment}** with an estimated price of **₹{pred_price:.2f} lakh**.
'''))

This activity covered data generation, exploration, preprocessing with pipelines, stratified train–test splitting, training and evaluating multiple classification and regression models, cross-validation, feature-importance analysis and prediction on a new sample. Because the price formula is largely linear, linear models perform very well; tree-based models are useful for capturing non-linear effects. Since the data is synthetic, the results demonstrate the ML process and are not real property valuations.